# 08_Cybersecurity_Features.ipynb
## FakeJobShield: Cybersecurity Indicator Engineering
This notebook designs and extracts custom cybersecurity indicators from job postings. These include detecting OTP requests, national identification requests (PAN, Aadhaar, Passport), financial details (Bank Account, Debit Card), and advance processing fees. It computes a combined `Sensitive_Data_Score`, suspicious keywords counts, and personal recruiter email checks, then merges them back into the main dataset.


In [1]:
import pandas as pd
import numpy as np
import re
import os

# Adjust working directory if run from notebooks folder
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')


In [2]:
# Load original raw dataset
df = pd.read_csv("data/fake_job_postings.csv")
print("Dataset columns:", df.columns.tolist())


Dataset columns: ['title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent', 'in_balanced_dataset']


In [3]:
# Define free email domains lists
FREE_EMAIL_DOMAINS = {
    "gmail.com", "yahoo.com", "hotmail.com", "outlook.com", 
    "aol.com", "icloud.com", "protonmail.com", "mail.com", "yandex.com"
}

# Suspicious keyword lists
URGENCY_KEYWORDS = [
    "urgent", "immediately", "act now", "limited slots", "only few positions",
    "guaranteed", "no interview", "instant hire", "send details now", "quick money", "easy money"
]


In [4]:
# Define cybersecurity extraction rules
def detect_pattern(text, pattern):
    if not isinstance(text, str):
        return 0
    return 1 if re.search(pattern, text, re.IGNORECASE) else 0

def calculate_cybersecurity_features(row):
    text = " ".join([
        str(row.get("title", "")),
        str(row.get("description", "")),
        str(row.get("requirements", "")),
        str(row.get("company_profile", "")),
        str(row.get("benefits", ""))
    ]).lower()
    
    req_otp = detect_pattern(text, r"\botp\b|one[- ]time[- ]password")
    req_aadhaar = detect_pattern(text, r"\baadhaar\b|\baadhar\b|uidai")
    req_pan = detect_pattern(text, r"\bpan\b|\bpan card\b|permanent account number")
    req_passport = detect_pattern(text, r"\bpassport\b")
    req_bank = detect_pattern(text, r"bank account|routing number|account number|routing no")
    req_card = detect_pattern(text, r"debit card|credit card|cvv|card number")
    req_reg_fee = detect_pattern(text, r"registration fee|processing fee|application fee|interview fee")
    req_proc_fee = detect_pattern(text, r"processing charge|equipment deposit|wire transfer|money deposit")
    
    sensitive_flags = [req_otp, req_aadhaar, req_pan, req_passport, req_bank, req_card, req_reg_fee, req_proc_fee]
    sensitive_data_score = sum(sensitive_flags) / len(sensitive_flags)
    
    urgency_count = sum(1 for kw in URGENCY_KEYWORDS if kw in text)
    
    has_personal_email = 0
    if re.search(r"\b[a-z0-9._%+-]+@(gmail|yahoo|hotmail|outlook|aol|icloud|protonmail|mail|yandex)\.com\b", text, re.I):
        has_personal_email = 1
    
    has_suspicious_url = 0
    if re.search(r"bit\.ly|tinyurl|goo\.gl|\.tk/", text, re.I):
        has_suspicious_url = 1
        
    return pd.Series({
        "req_otp": req_otp,
        "req_aadhaar": req_aadhaar,
        "req_pan": req_pan,
        "req_passport": req_passport,
        "req_bank": req_bank,
        "req_card": req_card,
        "req_reg_fee": req_reg_fee,
        "req_proc_fee": req_proc_fee,
        "sensitive_data_score": sensitive_data_score,
        "urgency_count": urgency_count,
        "has_personal_email": has_personal_email,
        "has_suspicious_url": has_suspicious_url
    })


In [5]:
# Apply feature engineering
print("Extracting cybersecurity features from job posts...")
sec_features = df.apply(calculate_cybersecurity_features, axis=1)
print("Extraction complete!")


Extracting cybersecurity features from job posts...


Extraction complete!


In [6]:
# Merge back with original dataset
cyber_df = pd.concat([df, sec_features], axis=1)
print("New dataset shape:", cyber_df.shape)
print("Sensitive data score summary statistics:")
print(cyber_df["sensitive_data_score"].describe())


New dataset shape: (17880, 30)
Sensitive data score summary statistics:
count    17880.000000
mean         0.012088
std          0.051253
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          0.250000
Name: sensitive_data_score, dtype: float64


In [7]:
# Save dataset
os.makedirs("data", exist_ok=True)
cyber_df.to_csv("data/cybersecurity_features.csv", index=False)
print("Saved cybersecurity enriched features dataset to 'data/cybersecurity_features.csv'")


Saved cybersecurity enriched features dataset to 'data/cybersecurity_features.csv'
